# 05 - Paper Results Reproduction

**Purpose.** Build a reproducibility ledger for the IEEE Access 2022 IDC ensemble paper.

**Research integrity rule.** Do not claim exact reproduction unless the dataset copy, split file, preprocessing, backbones, optimizer settings, and training budget are all recorded.

## Paper Reference

Eid Alkhaldi and Ehsan Salari, "Ensemble Optimization for Invasive Ductal Carcinoma (IDC) Classification Using Differential Cartesian Genetic Programming," IEEE Access, 2022. DOI: `10.1109/ACCESS.2022.3228176`.

In [ ]:
from pathlib import Path
import hashlib
import json

import pandas as pd

PROJECT = Path('..').resolve()
CONFIG = PROJECT / 'configs' / 'paper_2022_idc.json'
EXPERIMENT = PROJECT / 'artifacts' / 'paper_2022_idc' / 'experiment.json'

print('config:', CONFIG)
print('experiment:', EXPERIMENT)
print('experiment exists:', EXPERIMENT.exists())

## Configuration Fingerprint

This gives the config a stable identity. If the hash changes, you are not looking at the same experiment protocol.

In [ ]:
config_text = CONFIG.read_text()
config_hash = hashlib.sha256(config_text.encode('utf-8')).hexdigest()
print('SHA256:', config_hash)
config = json.loads(config_text)
display(pd.json_normalize(config, sep='.').T.rename(columns={0: 'value'}))

## Current Run Summary

In [ ]:
if not EXPERIMENT.exists():
    print('No paper-style experiment found yet. Run:')
    print('uv run dcpgann-train data/idc --config configs/paper_2022_idc.json --output artifacts/paper_2022_idc')
else:
    report = json.loads(EXPERIMENT.read_text())
    print('dataset:', report['paper']['dataset'])
    print('split sizes:', report['split_sizes'])
    print('backbones:', report['backbones'])
    current = pd.Series(report['optimized_ensemble_test_metrics'], name='current_run')
    display(current[['accuracy', 'balanced_accuracy', 'precision', 'recall_sensitivity', 'specificity', 'f1', 'roc_auc']].to_frame())

## Paper-Reported Numbers

Fill these values from the final published paper table before claiming a reproduction. Leaving them blank is better than inventing them.

In [ ]:
PAPER_REPORTED = {
    # IEEE Access 2022, Tables 10 and 11.
    'accuracy': 0.92874,
    'balanced_accuracy': 0.93671,
    'f1': 0.90490,
    'recall_sensitivity': 0.96360,
    'precision': 0.85294,
    'specificity': 0.90981,
}

if EXPERIMENT.exists():
    paper = pd.Series(PAPER_REPORTED, name='paper_reported')
    current = pd.Series(report['optimized_ensemble_test_metrics'], name='current_run')
    comparison = pd.concat([paper, current, current - paper], axis=1)
    comparison.columns = ['paper_reported', 'current_run', 'current_minus_paper']
    display(comparison)

## Reproduction Claim Template

Use language like this in the README or paper supplement:

> We reproduced the experimental pipeline using the public Breast Histopathology Images IDC dataset. The exact run configuration is stored in `configs/paper_2022_idc.json`, the deterministic split is stored in `artifacts/paper_2022_idc/splits.json`, and the final metrics are stored in `artifacts/paper_2022_idc/experiment.json`. Differences from the published result are reported explicitly.

That wording is honest, precise, and hard to attack.